# Chapter 46: Vision Transformers & Attention
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo
**Objective:** Memahami arsitektur Vision Transformer (ViT), membedah mekanisme ekstraksi patch dan embedding posisi, mengimplementasikan Scaled Dot-Product Self-Attention secara komputasional, serta menganalisis kompleksitas komputasi atensi.

### 1. Landasan Teori & Formulasi Self-Attention
Diberikan tensor token citra input $\mathbf{Z} \in \mathbb{R}^{N \times D}$, matriks Query, Key, dan Value diproyeksikan dengan:
$$\mathbf{Q} = \mathbf{Z}\mathbf{W}_Q, \quad \mathbf{K} = \mathbf{Z}\mathbf{W}_K, \quad \mathbf{V} = \mathbf{Z}\mathbf{W}_V$$

Mekanisme Scaled Dot-Product Attention dirumuskan:
$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Softmax}\left( \frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}} \right) \mathbf{V}$$
Kompleksitas komputasi berskala kuadrat terhadap jumlah token: $\mathcal{O}(N^2 \cdot D)$.

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import math
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    import torch.nn as nn
    import torchvision.models as models
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Beroperasi dalam Mode Analisis Algoritmik NumPy.")

In [ ]:
# 3. Partisi Patch Citra & Scaled Dot-Product Attention (NumPy Engine)
def count_patches(h, w, p):
    return (h * w) // (p * p)

def scaled_dot_product_attention(q, k, v):
    d_k = q.shape[-1]
    scores = np.matmul(q, k.T) / math.sqrt(d_k)
    e_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attn_weights = e_scores / np.sum(e_scores, axis=-1, keepdims=True)
    out = np.matmul(attn_weights, v)
    return out, attn_weights

# Uji coba numerik dengan 3 token, d_k = 4 (Contoh Kuliah Section 3)
q = np.array([[1.0, 0.0, 1.0, 0.0], [0.0, 2.0, 0.0, 1.0], [1.0, 1.0, 0.0, 0.0]], dtype=np.float64)
k = np.array([[1.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 2.0], [0.0, 0.0, 1.0, 1.0]], dtype=np.float64)
v = np.array([[10.0, 0.0], [0.0, 20.0], [5.0, 5.0]], dtype=np.float64)

out_token, attn_matrix = scaled_dot_product_attention(q, k, v)
print("Jumlah Patch untuk citra 224x224 (patch 16x16):", count_patches(224, 224, 16))
print("\nMatriks Bobot Perhatian (Softmax Attention Weights):\n", np.round(attn_matrix, 4))
print("\nVektor Representasi Teragregasi Token 0:\n", np.round(out_token[0], 4))

assert count_patches(224, 224, 16) == 196
assert np.isclose(attn_matrix[0, 0], 0.5065, atol=1e-3)
assert np.isclose(out_token[0, 0], 6.601, atol=1e-2)

In [ ]:
# 4. Analisis Skalabilitas Kompleksitas Komputasi O(N^2 * D)
patch_sizes = [32, 16, 8]
d_model = 768
img_res = 224

for p in patch_sizes:
    n_patches = (img_res // p) ** 2
    flops_attn = (n_patches ** 2) * d_model
    print(f"Patch {p}x{p} -> N={n_patches:4d} tokens -> Attn FLOPs: {flops_attn:,} ops")

In [ ]:
# 5. Real PyTorch Vision Transformer Workflow (ViT-Base Specification)
if HAS_TORCH:
    # Load pretrained Vision Transformer ViT-B/16 (lisensi BSD-3)
    vit_model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
    vit_model.eval()
    print("Pretrained Vision Transformer (ViT-B/16) Loaded Successfully.")
    print(f"Total Parameters: {sum(p.numel() for p in vit_model.parameters()):,}")
    
    # Forward pass citra sintetis (1, 3, 224, 224)
    x_dummy = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        logits = vit_model(x_dummy)
    print("ViT Real Inference Output Shape:", logits.shape)
else:
    print("Panduan Arsitektur Pretrained Vision Transformer (ViT-B/16):")
    print("1. Pretrained Checkpoint: torchvision.models.vit_b_16(weights=DEFAULT)")
    print("2. Input Tensor: Batch NCHW (B, 3, 224, 224) dinormalisasi mean=[0.485, 0.456, 0.406].")
    print("3. Dimensi Embedding: D = 768, 12 Encoder Blocks, 12 Attention Heads.")
    print("4. Jalankan pada Google Colab T4 GPU untuk visualisasi Attention Rollout peta atensi.")

### 6. Analisis Hasil, Evaluasi & Rekomendasi
1. **Global Receptive Field**: Mekanisme self-attention memungkinkan interaksi antar seluruh patch citra sejak lapisan awal, memberikan pemahaman konteks visual holistik.
2. **Bottleneck Resolusi Tinggi**: Kompleksitas kuadratik $\mathcal{O}(N^2)$ menuntut pemilihan ukuran patch yang bijaksana (misal $16 \times 16$) atau transisi ke arsitektur jendela lokal (Swin Transformer) untuk resolusi megapixel.
3. **Pemanfaatan Pretrained Model**: ViT membutuhkan data pretraining masif untuk mengompensasi ketiadaan bias induktif lokal, sehingga transfer learning dari checkpoint ImageNet-21k/1k adalah pendekatan standar terbaik.